In [ ]:
import pandas as pd

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run this notebook from the project root or notebooks directory')
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DICTIONARY_TRANSLATIONS_PATH, TRAIN_PARQUET_PATH, TRANSLATION_INFORMAL_PATH
import warnings
warnings.filterwarnings("ignore")

In [4]:
dictionnary_data = pd.read_json(DICTIONARY_TRANSLATIONS_PATH)
dictionnary_data

,wolof,français
0,Omar a lekk ceeb bi,C'est Omar qui a mangé le riz.
1,Man a,C'est moi.
2,Dama wax-a-wax ba sama bóli baa ngiy metti,J'ai tellement parlé que j’ai mal à la gorge.
3,Jénd naa tool bu réy-a-réy,J'ai acheté un très grand champ.
4,"Boo ko gisee muy dem ak a dikk nii rekk, dafa ...",quand tu le vois aller et venir comme ça sans ...
...,...,...
9621,Yéréem yaa ngi tooy xepp ak yuut,ses habits sont trempés de salive.
9622,Fu yuux yooyu jóge,D'où viennent ces cris ?
9623,Nit ñi dañu ko yuxoo,le public la sifflé.
9624,"Bul yuuxu, guddi na","ne crie pas, il fait nuit !"


In [8]:
len(dictionnary_data)

9626

In [4]:
translation_data = pd.read_parquet(TRAIN_PARQUET_PATH)
translation_data

,french,wolof,sources
0,Cette lettre doit être légalisée par le Minist...,Bataaxal bii jëwriñu lu ajju ci mbiru bitim ré...,flores
1,« On n’a pas eu le temps de se sauver.\n,"""Amuñu woon benn jot ngir rawal sunu bopp.\n",Microsoft Ntrex
2,L’Écosse est tout à fait votre maison et nous ...,"Ekost sa dëkk la te bëggna nu nga toog fi.""\n",Microsoft Ntrex
3,Jotnanews: L’Espagne injecte 84 milliards FCFA...,Xibaari Jotna : Espaañ joxe na juróom-ñett-fuk...,mafand
4,quel âge as-tu,ñaata at nga am,Google SMOL
...,...,...,...
17772,mettre en œuvre,taxawal,Google SMOL
17773,Les pays de la CEMAC refusent de reconduire le...,Réewi CEMAC yi dañu lànk a yeesalaat jokkalant...,mafand
17774,Karim Benzema s’est montré mais a manqué un pe...,"Karim Benzema wone na boppam, waaye dafa moy b...",mafand
17775,"Car, à ce minimum fiscal, il faut au moins y a...","Ndax, ci li gën a tuuti ci lempo bi, dañu ci w...",mafand


In [5]:
dictionnary_data = dictionnary_data[["wolof"]]
dictionnary_data.rename(columns={"wolof": "standard wolof"}, inplace=True)

translation_data = translation_data[["wolof"]]
translation_data.rename(columns={"wolof": "standard wolof"}, inplace=True)

In [6]:
import re

rules = [
    (r"(\w+)-a-(\w+)", r"\1a\2"), # Replace "word-a-word" with "wordaword"
    (r"-", r" "), # Replace hyphens with spaces
    (r"([aieo])\1", r"\1"),  # aa/ii/ee/oo -> a/i/e/o
    (r"uu",  r"__OU__"),          # uu -> ou

    (r"ñ+", r"gn"), # ñ -> gn
    (r"ŋ+", r"ng"), # ŋ -> ng
    (r"ë+", r"__EU__"),         # ë -> eu
    (r"ée", r"e"), # ée -> e
    (r"g([ieéèêë])", r"g__U__\1"), # g followed by i/e/é/è/ê/ë -> g + u + vowel
    (r"ó", r"o"), # ó -> o
    (r"(à|á|â|ä)", r"a"), # à/á/â/ä -> a
    (r"u([blt]+)", r"__OU__\1"), # u followed by b/l/t -> ou + b/l/t
    (r"u+", r"__OU__"), # u -> ou
    (r"q", r"kh"), # q -> kh
    (r"x", r"kh"), # x -> kh
    (r"c([aeiouy]+)", r"th\1"), # c followed by a vowel -> th + vowel
    (r"c{2}\b", r"thie"), # cc at the end of a word -> thie
    (r"[Jj]([eaoë]{1,2})", r"di\1"), # j followed by e/a/o/ë -> di + vowel
    (r"[Jj]([i]+)", r"dj\1"), # j followed by i -> dj + i
    (r"[Jj]([u]+)", r"dio\1"), # j followed by u -> dio + u
    (r"th([aeouy]+)", r"thi\1"), # th followed by a vowel -> thi + vowel
    (r"e\b", r"é"), # e at the end of a word -> é
    (r"n\b", r'ne'), # n at the end of a word -> ne
    (r"gg", r"g"), # gg -> g

    (r"__EU__", r"eu"),
    (r"__OU__", r"ou"),
    (r"__U__", r"u")
]

def transliterate(text: str) -> str:
    text = text.lower()
    for pattern, replacement in rules:
        text = re.sub(pattern, replacement, text)
    return text

In [7]:
from tqdm import tqdm
tqdm.pandas()
dictionnary_data["informal wolof"] = dictionnary_data["standard wolof"].progress_apply(transliterate)
dictionnary_data

100%|██████████| 9626/9626 [00:00<00:00, 45239.16it/s]


,standard wolof,informal wolof
0,Omar a lekk ceeb bi,omar a lekk thieb bi
1,Man a,mane a
2,Dama wax-a-wax ba sama bóli baa ngiy metti,dama wakhawakh ba sama boli ba nguiy metti
3,Jénd naa tool bu réy-a-réy,jénd na tol bou réyaréy
4,"Boo ko gisee muy dem ak a dikk nii rekk, dafa ...","bo ko guisé mouy dem ak a dikk ni rekk, dafa d..."
...,...,...
9621,Yéréem yaa ngi tooy xepp ak yuut,yérem ya ngui toy khepp ak yout
9622,Fu yuux yooyu jóge,fou youkh yoyou diogué
9623,Nit ñi dañu ko yuxoo,nit gni dagnou ko youkho
9624,"Bul yuuxu, guddi na","boul youkhou, gouddi na"


In [8]:
translation_data["informal wolof"] = translation_data["standard wolof"].progress_apply(transliterate)
translation_data

100%|██████████| 17777/17777 [00:00<00:00, 28455.42it/s]


,standard wolof,informal wolof
0,Bataaxal bii jëwriñu lu ajju ci mbiru bitim ré...,batakhal bi jeuwrignou lou ajjou thi mbirou bi...
1,"""Amuñu woon benn jot ngir rawal sunu bopp.\n","""amougnou wone benne diot nguir rawal sounou b..."
2,"Ekost sa dëkk la te bëggna nu nga toog fi.""\n","ekost sa deukk la té beugna nou nga tog fi.""\n"
3,Xibaari Jotna : Espaañ joxe na juróom-ñett-fuk...,khibari diotna : espagne diokhé na jouroom gne...
4,ñaata at nga am,gnata at nga am
...,...,...
17772,taxawal,takhawal
17773,Réewi CEMAC yi dañu lànk a yeesalaat jokkalant...,rewi thiemac yi dagnou lank a yesalat diokkala...
17774,"Karim Benzema wone na boppam, waaye dafa moy b...","karim benzema woné na boppam, wayé dafa moy be..."
17775,"Ndax, ci li gën a tuuti ci lempo bi, dañu ci w...","ndakh, thi li geune a touti thi lempo bi, dagn..."


In [21]:
import pandas as pd
import re
import warnings
from tqdm import tqdm
from transformers import pipeline

warnings.filterwarnings("ignore")

# ==========================================
# 1. LOAD DATA
# ==========================================
print("Loading datasets...")
# Using your exact paths
translation_data = pd.read_parquet(TRAIN_PARQUET_PATH)

translation_data = translation_data[["wolof","french"]].rename(columns={"wolof": "standard wolof"})


# ==========================================
# 2. INITIALIZE BATCHED NER PIPELINE
# ==========================================
print("Loading multilingual NER pipeline onto GPU...")
# This pipeline automatically detects and utilizes your GPU if configured
ner_pipeline = pipeline(
    "token-classification", 
    model="davlan/xlm-roberta-large-ner-hrl", 
    aggregation_strategy="simple",
    device=0 # Forces GPU usage (set to -1 if you want CPU only)
)


# ==========================================
# 3. REGEX TRANSLITERATION RULES
# ==========================================
rules = [
    (r"(\w+)-a-(\w+)", r"\1a\2"), (r"-", r" "), (r"([aieo])\1", r"\1"),  
    (r"uu",  r"__OU__"), (r"ñ+", r"gn"), (r"ŋ+", r"ng"), (r"ë+", r"__EU__"),  
    (r"ée", r"e"), (r"g([ieéèêë])", r"g__U__\1"), (r"ó", r"o"), (r"(à|á|â|ä)", r"a"), 
    (r"u([blt]+)", r"__OU__\1"), (r"u+", r"__OU__"), (r"q", r"kh"), (r"x", r"kh"), 
    (r"c([aeiouy]+)", r"th\1"), (r"c{2}\b", r"thie"), (r"[Jj]([eaoë]{1,2})", r"di\1"), 
    (r"[Jj]([i]+)", r"dj\1"), (r"[Jj]([u]+)", r"dio\1"), (r"th([aeouy]+)", r"thi\1"), 
    (r"e\b", r"é"), (r"n\b", r'ne'), (r"gg", r"g"), 
    (r"__EU__", r"eu"), (r"__OU__", r"ou"), (r"__U__", r"u")
]

def transliterate(text: str) -> str:
    text = text.lower()
    for pattern, replacement in rules:
        text = re.sub(pattern, replacement, text)
    return text


# ==========================================
# 4. CORE ENGINE: SAFE MASKING & TRANSLITERATION
# ==========================================
def process_safe_transliteration(text: str, entities: list) -> str:
    """
    Takes a single sentence and its pre-calculated entities.
    Automatically protects NER results, acronyms, and suspected French loan words
    without needing any hardcoded word lists.
    """
    if not isinstance(text, str) or not text.strip():
        return text

    protected_intervals = []
    
    # 1. Automatically grab whatever the NER model managed to find
    for ent in entities:
        protected_intervals.append((ent['start'], ent['end'], text[ent['start']:ent['end']]))

    # 2. Automated Structural Protection: 
    # Match any word that is entirely UPPERCASE (Acronyms like CEP, BCEAO, etc.)
    automated_patterns = r'\b[A-Z\d]{2,}\b'
    
    for match in re.finditer(automated_patterns, text):
        start, end = match.start(), match.end()
        # Ensure we don't duplicate boundaries if NER already caught it
        if not any(s <= start < e or s < end <= e for s, e, _ in protected_intervals):
            protected_intervals.append((start, end, text[start:end]))

    # 3. Sort intervals completely in reverse order 
    # This ensures string slicing index counts remain accurate
    protected_intervals = sorted(protected_intervals, key=lambda x: x[0], reverse=True)

    saved_entities = {}
    masked_text = text

    # Phase 1: Purely Numeric Masking (Immune to regex rules)
    for i, (start, end, original_val) in enumerate(protected_intervals):
        mask_token = f" 9999{i}9999 "
        saved_entities[mask_token.strip()] = original_val
        masked_text = masked_text[:start] + mask_token + masked_text[end:]

    # Phase 2: Run your phonetic rules loop safely
    informal_masked_text = transliterate(masked_text)

    # Phase 3: Targeted Precision Unmasking
    final_text = informal_masked_text
    for mask_token, original_value in saved_entities.items():
        final_text = final_text.replace(mask_token, original_value)

    # Clean up any residual double-spaces
    final_text = re.sub(r'\s+', ' ', final_text).strip()
    return final_text

# ==========================================
# 5. EFFICIENT BATCH PROCESSING PIPELINE
# ==========================================
def data_generator(df, column_name):
    """Generator function to stream text rows into the transformer pipeline efficiently."""
    for text in df[column_name]:
        yield str(text) if pd.notna(text) else ""

def run_pipeline(dataframe, target_column):
    """Runs batched NER extraction followed by safe local transliteration."""
    print(f"-> Step 1/2: Extracting entities in batches from '{target_column}'...")
    
    # batch_size=32 tells the GPU to process 32 rows concurrently, maximizing performance
    all_entities = list(tqdm(
        ner_pipeline(data_generator(dataframe, target_column), batch_size=32),
        total=len(dataframe)
    ))
    
    print("-> Step 2/2: Applying safe informalization rules...")
    final_results = []
    
    # Zip original texts and calculated entities together for fast sequential mapping
    for text, entities in tqdm(zip(dataframe[target_column], all_entities), total=len(dataframe)):
        processed_text = process_safe_transliteration(text, entities)
        final_results.append(processed_text)
        
    return final_results


# ==========================================
# 6. EXECUTION
# ==========================================

print("\n--- Processing Translation Data ---")
translation_data["informal wolof"] = run_pipeline(translation_data, "standard wolof")

# View a sneak peek of your safe, code-mixed informal output
print("\nSample Output:")
print(translation_data[["standard wolof", "informal wolof"]].head())

Loading datasets...
Loading multilingual NER pipeline onto GPU...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10025.82it/s]
XLMRobertaForTokenClassification LOAD REPORT from: davlan/xlm-roberta-large-ner-hrl
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Processing Translation Data ---
-> Step 1/2: Extracting entities in batches from 'standard wolof'...


100%|██████████| 17777/17777 [00:00<00:00, 5931753.56it/s]


-> Step 2/2: Applying safe informalization rules...


100%|██████████| 17777/17777 [00:00<00:00, 22692.83it/s]


Sample Output:
                                      standard wolof  \
0  Bataaxal bii jëwriñu lu ajju ci mbiru bitim ré...   
1       "Amuñu woon benn jot ngir rawal sunu bopp.\n   
2      Ekost sa dëkk la te bëggna nu nga toog fi."\n   
3  Xibaari Jotna : Espaañ joxe na juróom-ñett-fuk...   
4                                    ñaata at nga am   

                                      informal wolof  
0  batakhal bi jeuwrignou lou ajjou thi mbirou bi...  
1  "amougnou wone benne diot nguir rawal sounou b...  
2       ekost sa deukk la té beugna nou nga tog fi."  
3  Xibaari Jotna : Espaañ diokhé na jouroom gnett...  
4                                    gnata at nga am  


In [22]:
translation_data.iloc[17765].to_dict()

{'standard wolof': 'Ca Guinée Bissau ak ca Niger daanaka yeksi nañu ci... Yaakaar nañu ne itam coppitee yooyu dina eksi Senegaal ci atum 2024.\n',
 'french': 'En Guinée Bissau comme au Niger, nous y sommes presque…Nous espérons que ce vent de changement gagnera le Sénégal en 2024.\n',
 'informal wolof': 'thia Guinée Bissau ak thia Niger danaka yeksi nagnou thi... yakar nagnou né itam thioppité yoyou dina eksi Senegaal thi atoum 2024 .'}

In [23]:
translation_data.tail(20)

,standard wolof,french,informal wolof
17757,Fasoŋu Santaru Uragaan bu kawe bi defa ni dina...,Les dernières estimations du National Hurrican...,fasongou santarou our a gane bou kawé bi defa ...
17758,seddale,partager,seddalé
17759,Juróomi milyaar ci xaalisu Seefaa lañu jox fan...,5 milliards FCFA pour la promotion du financem...,Jur oomi milyar thi khalisou Seefaa lagnou dio...
17760,Aada yu tuuti yi yenn saay denuy seey te du nu...,Les cultures minoritaires disparaissent souven...,ada you touti yi yenne say denouy sey té dou n...
17761,xët fukk ak ñett bu nettalib CEP) Ki gàlloo ña...,page 13 du rapport de la CEP) Le principal res...,kheut foukk ak gnett bou nettalib CEP ) ki gal...
17762,Kowalisiyon ak Turupi Afganistaan yi toxu nañu...,Les troupes de coalition et Afghanes se sont d...,kowalisiyone ak touroupi afganistane yi tokhou...
17763,Su ma la laajee fan la iraañum bi ak peterolu ...,Si je pose ici la question de savoir où passen...,sou ma la ladié fane la iragnoum bi ak peterol...
17764,Lenn ci li gëna neex ci luuma yi mooy xumbaay bi.,L'une des meilleures choses sur les marchés en...,Len ne thi li geuna nekh thi louma yi moy khou...
17765,Ca Guinée Bissau ak ca Niger daanaka yeksi nañ...,"En Guinée Bissau comme au Niger, nous y sommes...",thia Guinée Bissau ak thia Niger danaka yeksi ...
17766,"Ci peggu ponk yi ñu teg war koo jëf, am na yen...","A côté de ce programme support, figurent des p...","thi pegou ponk yi gnou teg war ko jeuf, am na ..."


In [24]:
translation_data.to_csv(TRANSLATION_INFORMAL_PATH, index=False)